In [102]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json

In [122]:
def combine_subjects(df, codes_df):

    combined_df_data=[]

    for i, row in codes_df.iterrows():

        same_sub_mrns=[]

        for col in codes_df.columns:
            if col=='IncomingSite' or col=='Status':
                continue
            if np.isnan(row[col]):
                continue
            
            if int(row[col]) in df['MRN'].values and int(row[col]) not in same_sub_mrns:
                same_sub_mrns.append(int(row[col]))

        same_sub_df=df[df['MRN'].isin(same_sub_mrns)]

        if len(same_sub_mrns)==1:
            combined_df_data.append(same_sub_df.iloc[0].tolist()+[', '.join(str(mrn) for mrn in same_sub_mrns)])

        elif len(same_sub_mrns)>1:
            combined_row=[row['IncomingId']]
            for col in same_sub_df.columns[1:]:
                combined_row.append(same_sub_df[col].max())
            combined_df_data.append(combined_row+[', '.join(str(mrn) for mrn in same_sub_mrns)])


    out=pd.DataFrame(combined_df_data, columns=df.columns.tolist()+['other_matched_mrns'])
    out[df.columns.tolist()]=out[df.columns.tolist()].astype(int)

    return out


def accuracy(conf_matrix):
    
    total_questions=sum([sum(row) for row in conf_matrix])
    good_answers=0
    for i in range(len(conf_matrix)):
        good_answers += conf_matrix[i][i]
    accuracy=good_answers/total_questions
    return total_questions,accuracy

In [110]:
ccas_questions=list(json.load(open('/Users/rm026/Documents/code/reviewpyper/extraction_questions.json'))['ccas'].keys())
# ccas_questions.append("Is this patient able to identify similarities between objects, such as nose/ear, sheep/elephant, lake/river, or airplane/motorcycle?")
# ccas_questions.remove("Is this patient unable to identify similarities between objects, such as nose/ear, sheep/elephant, lake/river, or airplane/motorcycle?")
ai=pd.read_csv("/Users/rm026/Documents/schmahmann/schmahmann_output_w_unknown/master_list.csv")

ai=ai[['MRN']+ccas_questions]
gt=pd.read_csv("/Users/rm026/Documents/schmahmann/ground_truth_master_list_ccas_combined.csv")
# for stuff in zip(gt.columns.tolist()[:-1],ai.columns.tolist()):
#     print(stuff)
abbrev_questions=gt.columns.tolist()[1:-1]
ai.columns=['MRN']+abbrev_questions
ai=combine_subjects(ai, pd.read_csv('/Users/rm026/Partners HealthCare Dropbox/Ross Macfadyen/raynor_network_mapping/metadata/schmahmann_rpdr_request/rm026_110525111627789405_Mrn.txt', sep='|'))
ai.drop(columns=['other_matched_mrns'], inplace=True)
# gt.rename({'Does this patient have unusual affect? For instance, they may have difficulty with focusing attention or mental flexibility; be emotionally labile or show incongruous emotions; show easy sensory overload or avoidant behaviors; express illogical thoughts; lack empathy, be apathetic, or have blunted  or be angry or aggressive, irritable, oppositional, with difficulty with social cues and social boundaries. Ignore depression or anxiety symptoms.':"Does this patient have unusual affect? For instance, they may have difficulty with focusing attention or mental flexibility; be emotionally labile or show incongruous emotions; show easy sensory overload or avoidant behaviors; express illogical thoughts; lack empathy, be apathetic, or have blunted affect; or be angry or aggressive, irritable, oppositional, with difficulty with social cues and social boundaries. Ignore depression or anxiety symptoms."},axis=1,inplace=True)
merged=pd.merge(ai,gt,on='MRN',suffixes=('_ai','_gt'))
merged

,MRN,semantic fluency_ai,phonemic fluency_ai,category switching_ai,verbal registration_ai,digit span_ai,cube draw_ai,cube copy_ai,similarities_ai,go-no go_ai,...,phonemic fluency_gt,category switching_gt,verbal registration_gt,digit span_gt,cube draw_gt,cube copy_gt,similarities_gt,go-no go_gt,affect_gt,other_matched_mrns
0,6596307,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,"6596307, 26975813, 100122337"
1,6890459,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6890459
2,6868827,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,"6868827, 1418440"
3,7298779,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,7298779
4,8231339,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,8231339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,3699762,0,2,2,0,0,0,0,0,0,...,2,2,2,0,0,0,0,0,1,"3699762, 212623, 7111919"
81,3815202,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3815202
82,1175258,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1175258
83,1847722,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,"1847722, 7741061"


In [112]:
conf_matrix=[[0,0,0],
             [0,0,0],
             [0,0,0]]
for q in abbrev_questions:
    for truth, pred in zip(merged[q+'_gt'],merged[q+'_ai']):
        if type(pred)==str:
            print(f"Skipping string prediction: {pred}")
            continue
        conf_matrix[int(truth)][int(pred)]+=1
conf_matrix=np.array(conf_matrix)

array([[590,  16,   2],
       [  6, 132,   8],
       [  1,   1,  94]])

In [113]:
conf_df=pd.DataFrame(conf_matrix,columns=['Predict unknown','Predict no','Predict yes'],index=['Truth unknown','Truth no','Truth yes'])
conf_df

,Predict unknown,Predict no,Predict yes
Truth unknown,590,16,2
Truth no,6,132,8
Truth yes,1,1,94


In [115]:
accuracy(conf_matrix)

(np.int64(850), np.float64(0.96))

In [116]:
errors_of_omission=np.array([conf_matrix[0]+conf_matrix[1], conf_matrix[2]])
errors_of_omission=np.array([errors_of_omission[:,0]+errors_of_omission[:,1], errors_of_omission[:,2]])
# errors_of_omission['Pred no/unknown']=errors_of_omission['Predict unknown']+errors_of_omission['Predict no']
errors_of_omission, accuracy(errors_of_omission)

(array([[744,   2],
        [ 10,  94]]),
 (np.int64(850), np.float64(0.9858823529411764)))

In [118]:
no_unk=pd.read_csv("/Users/rm026/Documents/schmahmann/schmahmann_reviewpyper_dec_4/results/master_list.csv")
no_unk=no_unk[['MRN']+ccas_questions]
no_unk.columns=['MRN']+abbrev_questions
no_unk=combine_subjects(no_unk, pd.read_csv('/Users/rm026/Partners HealthCare Dropbox/Ross Macfadyen/raynor_network_mapping/metadata/schmahmann_rpdr_request/rm026_110525111627789405_Mrn.txt', sep='|'))
no_unk.drop(columns=['other_matched_mrns'], inplace=True)
for question in abbrev_questions:
    no_unk[question]=np.array(no_unk[question])+1
merged_no_unk=pd.merge(no_unk,gt,on='MRN',suffixes=('_ai','_gt'))
merged_no_unk

,MRN,semantic fluency_ai,phonemic fluency_ai,category switching_ai,verbal registration_ai,digit span_ai,cube draw_ai,cube copy_ai,similarities_ai,go-no go_ai,...,phonemic fluency_gt,category switching_gt,verbal registration_gt,digit span_gt,cube draw_gt,cube copy_gt,similarities_gt,go-no go_gt,affect_gt,other_matched_mrns
0,6596307,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,"6596307, 26975813, 100122337"
1,6890459,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,6890459
2,6868827,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,"6868827, 1418440"
3,7298779,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,7298779
4,8231339,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,8231339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,3699762,1,2,2,1,1,1,1,1,1,...,2,2,2,0,0,0,0,0,1,"3699762, 212623, 7111919"
81,3815202,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,3815202
82,1175258,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,1,1175258
83,1847722,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,2,"1847722, 7741061"


In [119]:
no_unk_matrix=[[0,0,0],
             [0,0,0],
             [0,0,0]]
for q in abbrev_questions:
    for truth, pred in zip(merged_no_unk[q+'_gt'],merged_no_unk[q+'_ai']):
        if type(pred)==str:
            print(f"Skipping string prediction: {pred}")
            continue
        no_unk_matrix[int(truth)][int(pred)]+=1
no_unk_matrix=np.array(no_unk_matrix)
no_unk_matrix=np.array([list(no_unk_matrix[0][1:]+no_unk_matrix[1][1:]), list(no_unk_matrix[2][1:])])
no_unk_matrix, accuracy(no_unk_matrix)

(array([[742,  12],
        [ 14,  82]]),
 (np.int64(850), np.float64(0.9694117647058823)))

In [120]:
conf_matrix

array([[590,  16,   2],
       [  6, 132,   8],
       [  1,   1,  94]])

In [121]:
ignoring_unknowns_accuracy=accuracy(conf_matrix[1:,1:])
ignoring_unknowns_accuracy

(np.int64(235), np.float64(0.9617021276595744))